In [1]:
library(Rcpp)
library(progress)
library(RcppEigen)
library(RcppDist)
library(RcppArmadillo)
library(mvtnorm)
library(dbarts)
sourceCpp("FirstModel.cpp")

Warning message:
"package 'Rcpp' was built under R version 4.3.3"
Warning message:
"package 'progress' was built under R version 4.3.3"
Warning message:
"package 'RcppEigen' was built under R version 4.3.3"
Warning message:
"package 'RcppDist' was built under R version 4.3.3"
Registered S3 methods overwritten by 'RcppArmadillo':
  method               from     
  predict.fastLm       RcppEigen
  print.fastLm         RcppEigen
  summary.fastLm       RcppEigen
  print.summary.fastLm RcppEigen


Attaching package: 'RcppArmadillo'


The following objects are masked from 'package:RcppEigen':

    fastLm, fastLmPure


Warning message:
"package 'mvtnorm' was built under R version 4.3.3"


# DGP_1

In [2]:
#Define Helper Functions
in_cred<-function(samples, value, interval)
{
  upper_quantile<-1-(1-interval)/2
  lower_quantile<-0+(1-interval)/2

  q1<-quantile(samples, lower_quantile)
  q2<-quantile(samples, upper_quantile)

  in_cred<-ifelse(value>=q1 & value<=q2, T, F)
}

cred_width<-function(samples, interval)
{
  upper_quantile<-1-(1-interval)/2
  lower_quantile<-0+(1-interval)/2

  q1<-quantile(samples, lower_quantile)
  q2<-quantile(samples, upper_quantile)

  return(q2-q1)
}

num_gfr<-0

# Number of simulations
num_simulations <- 100

# Initialize a matrix to store the results
new_colnames <- c(
  # PEHE metrics
  "mvbcf_1k_pehe1", "mvbcf_1k_pehe2",
  "mvbcf_0.5k_pehe1", "mvbcf_0.5k_pehe2",
  "mvbcf_0.25k_pehe1", "mvbcf_0.25k_pehe2",
  "mvbcf_0.1k_pehe1", "mvbcf_0.1k_pehe2",
  "mvbcf_0.05k_pehe1", "mvbcf_0.05k_pehe2",

  # RMSE metrics (added)
  "mvbcf_1k_rmse1", "mvbcf_1k_rmse2",
  "mvbcf_0.5k_rmse1", "mvbcf_0.5k_rmse2",
  "mvbcf_0.25k_rmse1", "mvbcf_0.25k_rmse2",
  "mvbcf_0.1k_rmse1", "mvbcf_0.1k_rmse2",
  "mvbcf_0.05k_rmse1", "mvbcf_0.05k_rmse2",

  # MAPE metrics (added)
  "mvbcf_1k_mape1", "mvbcf_1k_mape2",
  "mvbcf_0.5k_mape1", "mvbcf_0.5k_mape2",
  "mvbcf_0.25k_mape1", "mvbcf_0.25k_mape2",
  "mvbcf_0.1k_mape1", "mvbcf_0.1k_mape2",
  "mvbcf_0.05k_mape1", "mvbcf_0.05k_mape2",

  # Tau 95% interval width metrics
  "mvbcf_1k_tau_951", "mvbcf_1k_tau_952",
  "mvbcf_0.5k_tau_951", "mvbcf_0.5k_tau_952",
  "mvbcf_0.25k_tau_951", "mvbcf_0.25k_tau_952",
  "mvbcf_0.1k_tau_951", "mvbcf_0.1k_tau_952",
  "mvbcf_0.05k_tau_951", "mvbcf_0.05k_tau_952",

  # Tau 95% interval width metrics (weighted)
  "mvbcf_1k_tau_951w", "mvbcf_1k_tau_952w",
  "mvbcf_0.5k_tau_951w", "mvbcf_0.5k_tau_952w",
  "mvbcf_0.25k_tau_951w", "mvbcf_0.25k_tau_952w",
  "mvbcf_0.1k_tau_951w", "mvbcf_0.1k_tau_952w",
  "mvbcf_0.05k_tau_951w", "mvbcf_0.05k_tau_952w"
)

# Calculate the total number of columns
num_columns <- length(new_colnames)

# Initialize the results matrix with the correct number of columns
results_matrix <- matrix(NA, nrow = num_simulations, ncol = num_columns)
colnames(results_matrix) <- new_colnames

# Create a progress bar
pb <- progress_bar$new(total = num_simulations)

# For loop to run the code 100 times
for (i in 1:num_simulations) {
  # Update progress bar
  pb$tick()
#Set random seed
seed_val<-i
set.seed(seed_val)

#Train Data
n<-500

X1<-rnorm(n)
X2<-rnorm(n)
X3<-rnorm(n)
X4<-rbinom(n, size = 1, prob = 0.5)
X5<-sample(1:3, size = n, replace = TRUE)

g_values <- c(2, -1, -4) # g(1)=2, g(2)=-1, g(3)=-4
g_x5 <- g_values[X5]

X<-cbind(X1, X2, X3, X4, X5)

Mu1<- -6 + g_x5 + 6 * abs(X3 - 1)+ X1 * X3
Mu2<- -4 + 0.66*g_x5 + 8.5 * abs(X3 - 0.85)+ 0.25*X1 * X3

Tau1<- 1 + 2 * X2 * X4
Tau2<- -1 + 3.5 * X2 * X4

s_linear <- sd(Mu1)

pi <- 0.8 * pnorm(3 * Mu1 / s_linear - 0.5 * X1) + 0.05 + runif(n) / 10

true_propensity<-pmin(1, pmax(0, pi))

Z<-rbinom(n, 1, true_propensity)

Y<-cbind(Mu1+Z*Tau1, Mu2+Z*Tau2) + mvtnorm::rmvnorm(n, c(0, 0), matrix(c(1, 0, 0, 1), nrow=2, byrow=T))

#Test Data
n_test<-1000

X1_test<-rnorm(n_test)
X2_test<-rnorm(n_test)
X3_test<-rnorm(n_test)
X4_test<-rbinom(n_test, size = 1, prob = 0.5)
X5_test<-sample(1:3, size = n_test, replace = TRUE)

g_x5_test <- g_values[X5_test]

X_test<-cbind(X1_test, X2_test, X3_test, X4_test, X5_test)

Mu1_test<- -6 + g_x5_test + 6 * abs(X3_test - 1)+ X1_test * X3_test
Mu2_test<- -4 + 0.66*g_x5_test + 8.5 * abs(X3_test - 0.85)+ 0.25*X1_test * X3_test

Tau1_test<- 1 + 2 * X2_test * X4_test
Tau2_test<- -1 + 3.5 * X2_test * X4_test

s_linear_test <- sd(Mu1_test)

pi_test <- 0.8 * pnorm(3 * Mu1_test / s_linear_test - 0.5 * X1_test) + 0.05 + runif(n_test) / 10

true_propensity_test<-pmin(1, pmax(0, pi_test))

Z_test<-rbinom(n_test, 1, true_propensity_test)

Y_test<-cbind(Mu1_test+Z_test*Tau1_test, Mu2_test+Z_test*Tau2_test) + mvtnorm::rmvnorm(n_test, c(0, 0), matrix(c(1, 0, 0, 1), nrow=2, byrow=T))

#estimate of propensity score
p_mod<-bart(x.train = X, y.train = Z, x.test = X_test, k=3, verbose = FALSE)
p<-colMeans(pnorm(p_mod$yhat.train))
p_test<-colMeans(pnorm(p_mod$yhat.test))

#adding to matrix
X2<-X
X2_test<-X_test
X<-cbind(X, p)
X_test<-cbind(X_test, p_test)
Z2<-cbind(Z,Z)

#set some parameters
n_tree_mu<-50
n_tree_tau<-20
n_iter<-1000
n_burn<-500

mu_val<-1
tau_val<-0.375
v_val<-1
wish_val<-1
min_val<-1

mvbcf_1k_mod <- fast_bart(X,
                         Y,
                         Z2,
                         X2,
                         X_test, # here is the test data for the mu part of the model
                         X2_test, # here is the test data for the tau part of the model
                         0.95,
                         2,
                         0.25,
                         3,
                         diag((mu_val)^2/n_tree_mu, 2),
                         diag((tau_val)^2/n_tree_tau, 2),
                         v_val,
                         diag(wish_val, 2),
                         n_iter,
                         n_tree_mu,
                         n_tree_tau,
                         min_val,
                         num_gfr)


mvbcf_1k_tau_preds1<-rowMeans(mvbcf_1k_mod$predictions_tau_test[,1,-c(1:n_burn)])
mvbcf_1k_ate1<-mean(mvbcf_1k_tau_preds1)
mvbcf_1k_tau_preds2<-rowMeans(mvbcf_1k_mod$predictions_tau_test[,2,-c(1:n_burn)])
mvbcf_1k_ate2<-mean(mvbcf_1k_tau_preds2)

mvbcf_1k_pehe1<-sqrt(mean((Tau1_test-mvbcf_1k_tau_preds1)^2))
mvbcf_1k_pehe2<-sqrt(mean((Tau2_test-mvbcf_1k_tau_preds2)^2))

ate1 <- mean(Tau1_test)
ate2 <- mean(Tau2_test)

mvbcf_1k_tau_951<-mean(diag(apply(mvbcf_1k_mod$predictions_tau_test[,1,-c(1:n_burn)], 1, in_cred, Tau1_test, 0.95)))
mvbcf_1k_tau_951w<-mean(apply(mvbcf_1k_mod$predictions_tau_test[,1,-c(1:n_burn)], 1, cred_width, 0.95))

mvbcf_1k_tau_952<-mean(diag(apply(mvbcf_1k_mod$predictions_tau_test[,2,-c(1:n_burn)], 1, in_cred, Tau2_test, 0.95)))
mvbcf_1k_tau_952w<-mean(apply(mvbcf_1k_mod$predictions_tau_test[,2,-c(1:n_burn)], 1, cred_width, 0.95))

mvbcf_1k_rmse1 <- sqrt(mean((Y_test[, 1] - mvbcf_1k_mod$predictions_test[,1,-c(1:n_burn)])^2))
mvbcf_1k_mape1 <- mean((abs(Y_test[, 1] - mvbcf_1k_mod$predictions_test[,2,-c(1:n_burn)]))/abs(Y_test[, 1]))

mvbcf_1k_rmse2 <- sqrt(mean((Y_test[, 2] - mvbcf_1k_mod$predictions_test[,1,-c(1:n_burn)])^2))
mvbcf_1k_mape2 <- mean((abs(Y_test[, 2] - mvbcf_1k_mod$predictions_test[,2,-c(1:n_burn)]))/abs(Y_test[, 2]))

n_iter<-500
n_burn<-250

mvbcf_0.5k_mod <- fast_bart(X,
                         Y,
                         Z2,
                         X2,
                         X_test, # here is the test data for the mu part of the model
                         X2_test, # here is the test data for the tau part of the model
                         0.95,
                         2,
                         0.25,
                         3,
                         diag((mu_val)^2/n_tree_mu, 2),
                         diag((tau_val)^2/n_tree_tau, 2),
                         v_val,
                         diag(wish_val, 2),
                         n_iter,
                         n_tree_mu,
                         n_tree_tau,
                         min_val,
                         num_gfr)


mvbcf_0.5k_tau_preds1<-rowMeans(mvbcf_0.5k_mod$predictions_tau_test[,1,-c(1:n_burn)])
mvbcf_0.5k_ate1<-mean(mvbcf_0.5k_tau_preds1)
mvbcf_0.5k_tau_preds2<-rowMeans(mvbcf_0.5k_mod$predictions_tau_test[,2,-c(1:n_burn)])
mvbcf_0.5k_ate2<-mean(mvbcf_0.5k_tau_preds2)

mvbcf_0.5k_pehe1<-sqrt(mean((Tau1_test-mvbcf_0.5k_tau_preds1)^2))
mvbcf_0.5k_pehe2<-sqrt(mean((Tau2_test-mvbcf_0.5k_tau_preds2)^2))

ate1 <- mean(Tau1_test)
ate2 <- mean(Tau2_test)

mvbcf_0.5k_tau_951<-mean(diag(apply(mvbcf_0.5k_mod$predictions_tau_test[,1,-c(1:n_burn)], 1, in_cred, Tau1_test, 0.95)))
mvbcf_0.5k_tau_951w<-mean(apply(mvbcf_0.5k_mod$predictions_tau_test[,1,-c(1:n_burn)], 1, cred_width, 0.95))

mvbcf_0.5k_tau_952<-mean(diag(apply(mvbcf_0.5k_mod$predictions_tau_test[,2,-c(1:n_burn)], 1, in_cred, Tau2_test, 0.95)))
mvbcf_0.5k_tau_952w<-mean(apply(mvbcf_0.5k_mod$predictions_tau_test[,2,-c(1:n_burn)], 1, cred_width, 0.95))

mvbcf_0.5k_rmse1 <- sqrt(mean((Y_test[, 1] - mvbcf_0.5k_mod$predictions_test[,1,-c(1:n_burn)])^2))
mvbcf_0.5k_mape1 <- mean((abs(Y_test[, 1] - mvbcf_0.5k_mod$predictions_test[,2,-c(1:n_burn)]))/abs(Y_test[, 1]))

mvbcf_0.5k_rmse2 <- sqrt(mean((Y_test[, 2] - mvbcf_0.5k_mod$predictions_test[,1,-c(1:n_burn)])^2))
mvbcf_0.5k_mape2 <- mean((abs(Y_test[, 2] - mvbcf_0.5k_mod$predictions_test[,2,-c(1:n_burn)]))/abs(Y_test[, 2]))

n_iter<-250
n_burn<-125

mvbcf_0.25k_mod <- fast_bart(X,
                         Y,
                         Z2,
                         X2,
                         X_test, # here is the test data for the mu part of the model
                         X2_test, # here is the test data for the tau part of the model
                         0.95,
                         2,
                         0.25,
                         3,
                         diag((mu_val)^2/n_tree_mu, 2),
                         diag((tau_val)^2/n_tree_tau, 2),
                         v_val,
                         diag(wish_val, 2),
                         n_iter,
                         n_tree_mu,
                         n_tree_tau,
                         min_val,
                         num_gfr)


mvbcf_0.25k_tau_preds1<-rowMeans(mvbcf_0.25k_mod$predictions_tau_test[,1,-c(1:n_burn)])
mvbcf_0.25k_ate1<-mean(mvbcf_0.25k_tau_preds1)
mvbcf_0.25k_tau_preds2<-rowMeans(mvbcf_0.25k_mod$predictions_tau_test[,2,-c(1:n_burn)])
mvbcf_0.25k_ate2<-mean(mvbcf_0.25k_tau_preds2)

mvbcf_0.25k_pehe1<-sqrt(mean((Tau1_test-mvbcf_0.25k_tau_preds1)^2))
mvbcf_0.25k_pehe2<-sqrt(mean((Tau2_test-mvbcf_0.25k_tau_preds2)^2))

ate1 <- mean(Tau1_test)
ate2 <- mean(Tau2_test)

mvbcf_0.25k_tau_951<-mean(diag(apply(mvbcf_0.25k_mod$predictions_tau_test[,1,-c(1:n_burn)], 1, in_cred, Tau1_test, 0.95)))
mvbcf_0.25k_tau_951w<-mean(apply(mvbcf_0.25k_mod$predictions_tau_test[,1,-c(1:n_burn)], 1, cred_width, 0.95))

mvbcf_0.25k_tau_952<-mean(diag(apply(mvbcf_0.25k_mod$predictions_tau_test[,2,-c(1:n_burn)], 1, in_cred, Tau2_test, 0.95)))
mvbcf_0.25k_tau_952w<-mean(apply(mvbcf_0.25k_mod$predictions_tau_test[,2,-c(1:n_burn)], 1, cred_width, 0.95))

mvbcf_0.25k_rmse1 <- sqrt(mean((Y_test[, 1] - mvbcf_0.25k_mod$predictions_test[,1,-c(1:n_burn)])^2))
mvbcf_0.25k_mape1 <- mean((abs(Y_test[, 1] - mvbcf_0.25k_mod$predictions_test[,2,-c(1:n_burn)]))/abs(Y_test[, 1]))

mvbcf_0.25k_rmse2 <- sqrt(mean((Y_test[, 2] - mvbcf_0.25k_mod$predictions_test[,1,-c(1:n_burn)])^2))
mvbcf_0.25k_mape2 <- mean((abs(Y_test[, 2] - mvbcf_0.25k_mod$predictions_test[,2,-c(1:n_burn)]))/abs(Y_test[, 2]))


n_iter<-100
n_burn<-50

mvbcf_0.1k_mod <- fast_bart(X,
                         Y,
                         Z2,
                         X2,
                         X_test, # here is the test data for the mu part of the model
                         X2_test, # here is the test data for the tau part of the model
                         0.95,
                         2,
                         0.25,
                         3,
                         diag((mu_val)^2/n_tree_mu, 2),
                         diag((tau_val)^2/n_tree_tau, 2),
                         v_val,
                         diag(wish_val, 2),
                         n_iter,
                         n_tree_mu,
                         n_tree_tau,
                         min_val,
                         num_gfr)


mvbcf_0.1k_tau_preds1<-rowMeans(mvbcf_0.1k_mod$predictions_tau_test[,1,-c(1:n_burn)])
mvbcf_0.1k_ate1<-mean(mvbcf_0.1k_tau_preds1)
mvbcf_0.1k_tau_preds2<-rowMeans(mvbcf_0.1k_mod$predictions_tau_test[,2,-c(1:n_burn)])
mvbcf_0.1k_ate2<-mean(mvbcf_0.1k_tau_preds2)

mvbcf_0.1k_pehe1<-sqrt(mean((Tau1_test-mvbcf_0.1k_tau_preds1)^2))
mvbcf_0.1k_pehe2<-sqrt(mean((Tau2_test-mvbcf_0.1k_tau_preds2)^2))

ate1 <- mean(Tau1_test)
ate2 <- mean(Tau2_test)

mvbcf_0.1k_tau_951<-mean(diag(apply(mvbcf_0.1k_mod$predictions_tau_test[,1,-c(1:n_burn)], 1, in_cred, Tau1_test, 0.95)))
mvbcf_0.1k_tau_951w<-mean(apply(mvbcf_0.1k_mod$predictions_tau_test[,1,-c(1:n_burn)], 1, cred_width, 0.95))

mvbcf_0.1k_tau_952<-mean(diag(apply(mvbcf_0.1k_mod$predictions_tau_test[,2,-c(1:n_burn)], 1, in_cred, Tau2_test, 0.95)))
mvbcf_0.1k_tau_952w<-mean(apply(mvbcf_0.1k_mod$predictions_tau_test[,2,-c(1:n_burn)], 1, cred_width, 0.95))

mvbcf_0.1k_rmse1 <- sqrt(mean((Y_test[, 1] - mvbcf_0.1k_mod$predictions_test[,1,-c(1:n_burn)])^2))
mvbcf_0.1k_mape1 <- mean((abs(Y_test[, 1] - mvbcf_0.1k_mod$predictions_test[,2,-c(1:n_burn)]))/abs(Y_test[, 1]))

mvbcf_0.1k_rmse2 <- sqrt(mean((Y_test[, 2] - mvbcf_0.1k_mod$predictions_test[,1,-c(1:n_burn)])^2))
mvbcf_0.1k_mape2 <- mean((abs(Y_test[, 2] - mvbcf_0.1k_mod$predictions_test[,2,-c(1:n_burn)]))/abs(Y_test[, 2]))



n_iter<-50
n_burn<-25

mvbcf_0.05k_mod <- fast_bart(X,
                         Y,
                         Z2,
                         X2,
                         X_test, # here is the test data for the mu part of the model
                         X2_test, # here is the test data for the tau part of the model
                         0.95,
                         2,
                         0.25,
                         3,
                         diag((mu_val)^2/n_tree_mu, 2),
                         diag((tau_val)^2/n_tree_tau, 2),
                         v_val,
                         diag(wish_val, 2),
                         n_iter,
                         n_tree_mu,
                         n_tree_tau,
                         min_val,
                         num_gfr)


mvbcf_0.05k_tau_preds1<-rowMeans(mvbcf_0.05k_mod$predictions_tau_test[,1,-c(1:n_burn)])
mvbcf_0.05k_ate1<-mean(mvbcf_0.05k_tau_preds1)
mvbcf_0.05k_tau_preds2<-rowMeans(mvbcf_0.05k_mod$predictions_tau_test[,2,-c(1:n_burn)])
mvbcf_0.05k_ate2<-mean(mvbcf_0.05k_tau_preds2)

mvbcf_0.05k_pehe1<-sqrt(mean((Tau1_test-mvbcf_0.05k_tau_preds1)^2))
mvbcf_0.05k_pehe2<-sqrt(mean((Tau2_test-mvbcf_0.05k_tau_preds2)^2))

ate1 <- mean(Tau1_test)
ate2 <- mean(Tau2_test)

mvbcf_0.05k_tau_951<-mean(diag(apply(mvbcf_0.05k_mod$predictions_tau_test[,1,-c(1:n_burn)], 1, in_cred, Tau1_test, 0.95)))
mvbcf_0.05k_tau_951w<-mean(apply(mvbcf_0.05k_mod$predictions_tau_test[,1,-c(1:n_burn)], 1, cred_width, 0.95))

mvbcf_0.05k_tau_952<-mean(diag(apply(mvbcf_0.05k_mod$predictions_tau_test[,2,-c(1:n_burn)], 1, in_cred, Tau2_test, 0.95)))
mvbcf_0.05k_tau_952w<-mean(apply(mvbcf_0.05k_mod$predictions_tau_test[,2,-c(1:n_burn)], 1, cred_width, 0.95))

mvbcf_0.05k_rmse1 <- sqrt(mean((Y_test[, 1] - mvbcf_0.05k_mod$predictions_test[,1,-c(1:n_burn)])^2))
mvbcf_0.05k_mape1 <- mean((abs(Y_test[, 1] - mvbcf_0.05k_mod$predictions_test[,2,-c(1:n_burn)]))/abs(Y_test[, 1]))

mvbcf_0.05k_rmse2 <- sqrt(mean((Y_test[, 2] - mvbcf_0.05k_mod$predictions_test[,1,-c(1:n_burn)])^2))
mvbcf_0.05k_mape2 <- mean((abs(Y_test[, 2] - mvbcf_0.05k_mod$predictions_test[,2,-c(1:n_burn)]))/abs(Y_test[, 2]))



# Store the results in the matrix
results_matrix[i, ] <- c(
  # PEHE metrics
  mvbcf_1k_pehe1, mvbcf_1k_pehe2,
  mvbcf_0.5k_pehe1, mvbcf_0.5k_pehe2,
  mvbcf_0.25k_pehe1, mvbcf_0.25k_pehe2,
  mvbcf_0.1k_pehe1, mvbcf_0.1k_pehe2,
  mvbcf_0.05k_pehe1, mvbcf_0.05k_pehe2,

  # RMSE metrics
  mvbcf_1k_rmse1, mvbcf_1k_rmse2,
  mvbcf_0.5k_rmse1, mvbcf_0.5k_rmse2,
  mvbcf_0.25k_rmse1, mvbcf_0.25k_rmse2,
  mvbcf_0.1k_rmse1, mvbcf_0.1k_rmse2,
  mvbcf_0.05k_rmse1, mvbcf_0.05k_rmse2,

  # MAPE metrics
  mvbcf_1k_mape1, mvbcf_1k_mape2,
  mvbcf_0.5k_mape1, mvbcf_0.5k_mape2,
  mvbcf_0.25k_mape1, mvbcf_0.25k_mape2,
  mvbcf_0.1k_mape1, mvbcf_0.1k_mape2,
  mvbcf_0.05k_mape1, mvbcf_0.05k_mape2,

  # Tau 95% interval width metrics
  mvbcf_1k_tau_951, mvbcf_1k_tau_952,
  mvbcf_0.5k_tau_951, mvbcf_0.5k_tau_952,
  mvbcf_0.25k_tau_951, mvbcf_0.25k_tau_952,
  mvbcf_0.1k_tau_951, mvbcf_0.1k_tau_952,
  mvbcf_0.05k_tau_951, mvbcf_0.05k_tau_952,

  # Tau 95% interval width metrics (weighted)
  mvbcf_1k_tau_951w, mvbcf_1k_tau_952w,
  mvbcf_0.5k_tau_951w, mvbcf_0.5k_tau_952w,
  mvbcf_0.25k_tau_951w, mvbcf_0.25k_tau_952w,
  mvbcf_0.1k_tau_951w, mvbcf_0.1k_tau_952w,
  mvbcf_0.05k_tau_951w, mvbcf_0.05k_tau_952w
)


}

# Export the results matrix to a CSV file
write.csv(results_matrix, "MVBCF_simulation_results_DGP1.csv", row.names = FALSE)

# Print a message indicating completion
cat("Simulation completed and results saved to MVBCF_simulation_results_DGP1.csv\n")

Warning message in validateXTest(test, attr(x, "term.labels"), ncol(x), colnames(x), :
"column names of 'test' does not equal that of 'x': 'X1, X2, X3, X4, X5'; match will be made by position"



Total MVBCF runtime: 31102 ms

Total MVBCF runtime: 12968 ms

Total MVBCF runtime: 6478 ms

Total MVBCF runtime: 2531 ms

Total MVBCF runtime: 1264 ms


Warning message in validateXTest(test, attr(x, "term.labels"), ncol(x), colnames(x), :
"column names of 'test' does not equal that of 'x': 'X1, X2, X3, X4, X5'; match will be made by position"



Total MVBCF runtime: 26411 ms

Total MVBCF runtime: 13135 ms

Total MVBCF runtime: 6551 ms

Total MVBCF runtime: 2570 ms

Total MVBCF runtime: 1253 ms


Warning message in validateXTest(test, attr(x, "term.labels"), ncol(x), colnames(x), :
"column names of 'test' does not equal that of 'x': 'X1, X2, X3, X4, X5'; match will be made by position"


...


Total MVBCF runtime: 26189 ms

Total MVBCF runtime: 13218 ms

Total MVBCF runtime: 6651 ms

Total MVBCF runtime: 2593 ms

Total MVBCF runtime: 1279 ms


Warning message in validateXTest(test, attr(x, "term.labels"), ncol(x), colnames(x), :
"column names of 'test' does not equal that of 'x': 'X1, X2, X3, X4, X5'; match will be made by position"



Total MVBCF runtime: 26257 ms

Total MVBCF runtime: 13045 ms

Total MVBCF runtime: 6359 ms

Total MVBCF runtime: 2663 ms

Total MVBCF runtime: 1275 ms


Warning message in validateXTest(test, attr(x, "term.labels"), ncol(x), colnames(x), :
"column names of 'test' does not equal that of 'x': 'X1, X2, X3, X4, X5'; match will be made by position"



Total MVBCF runtime: 26500 ms

Total MVBCF runtime: 13169 ms

Total MVBCF runtime: 6569 ms

Total MVBCF runtime: 2602 ms

Total MVBCF runtime: 1297 ms
Simulation completed and results saved to MVBCF_simulation_results_DGP1.csv


In [3]:
print(results_matrix)

       mvbcf_1k_pehe1 mvbcf_1k_pehe2 mvbcf_0.5k_pehe1 mvbcf_0.5k_pehe2
  [1,]      0.6868118      0.9882028        0.6249213        0.9302781
  [2,]      0.4652541      0.7311966        0.5936029        0.8552121
  [3,]      0.5729583      0.9041007        0.6845838        1.0286971
  [4,]      0.5630282      0.8305848        0.5728680        0.7017648
  [5,]      0.5534633      0.8491203        0.5276485        0.8147375
  [6,]      0.4742976      0.7468831        0.4817685        0.6924800
  [7,]      0.6073703      0.9602023        0.6520027        1.0947659
  [8,]      0.5076717      0.8883813        0.5697944        0.8908868
  [9,]      0.5801540      0.8661847        0.6826582        1.0945645
 [10,]      0.7780510      1.1960213        0.7207341        1.0615909
 [11,]      0.4848337      0.8531351        0.4894161        0.8124045
 [12,]      0.5115538      0.8731941        0.4896464        0.8948761
 [13,]      0.4468217      0.7645503        0.7355266        1.3134976
 [14,]